# EcoSort: EfficientNet-B0 vs current ResNet18
Select a T4 GPU, run the one code cell, and upload whichever files it requests. Keep **EcoSort_TrashNet_split.zip** and **EcoSort_RealWaste_Result.zip** available; select both together if both are requested. Existing dataset caches are reused when available.

Exact source/class/split membership is verified against the previous result before training. The candidate uses the same six classes and saves a separate EfficientNet checkpoint. It trains for up to 15 epochs, selects using validation and compares source-specific results with the latest ResNet18. No improvement is guaranteed. The four known internet photos remain diagnostic-only.

Sources and attribution: RealWaste (Single et al., 2023; CC BY 4.0), TACO (Proença and Simões, 2020), and Torchvision EfficientNet-B0 (Tan and Le, 2019). Full method and source URLs are included in the result README. This is not a completed GPU training run.

In [ ]:
# Controlled EfficientNet-B0 experiment for one main garbage item per photo.
from pathlib import Path
import sys, json, zipfile, shutil, hashlib, traceback, importlib.metadata, time
import torch
from PIL import Image
from google.colab import files
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > T4 GPU first.')
work=Path('/content/ecosort_effnet_work');work.mkdir(parents=True,exist_ok=True)
source=work/'Src';source.mkdir(exist_ok=True)
MODULES={'taco_data': '"""TACO plastic-packaging crops, with source-grouped splits and provenance."""\nimport concurrent.futures\nimport hashlib\nimport io\nimport json\nimport math\nimport random\nimport urllib.request\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom PIL import Image, ImageDraw, ImageOps\n\nTACO_COMMIT = \'70479ad9adc3da971a8ae75ac97eb1026910eec7\'\nANNOTATIONS_URL = f\'https://raw.githubusercontent.com/pedropro/TACO/{TACO_COMMIT}/data/annotations.json\'\nCATEGORIES = {\'Plastic film\', \'Other plastic wrapper\', \'Crisp packet\'}\nCLASSES = [\'cardboard\', \'glass\', \'metal\', \'paper\', \'plastic\', \'trash\']\nPOLICY = (\'EcoSort plastic includes visually identified flexible plastic packaging, including \'\n          \'metallized snack packets, following the project packaging convention. \'\n          \'Source labels do not verify chemical composition or recyclability.\')\n\ndef sha256(data):\n    return hashlib.sha256(data).hexdigest()\n\ndef pixel_hash(image):\n    rgb = image.convert(\'RGB\')\n    return sha256(str(rgb.size).encode() + rgb.tobytes())\n\ndef dhash(image):\n    small = image.convert(\'L\').resize((9, 8), Image.Resampling.LANCZOS)\n    pixels = list(small.getdata())\n    result = 0\n    for y in range(8):\n        for x in range(8):\n            result = (result << 1) | (pixels[y*9+x] > pixels[y*9+x+1])\n    return result\n\ndef crop_box(bbox, original_size, downloaded_size):\n    x, y, width, height = bbox\n    sx, sy = downloaded_size[0]/original_size[0], downloaded_size[1]/original_size[1]\n    return (max(0, math.floor(x*sx)), max(0, math.floor(y*sy)),\n            min(downloaded_size[0], math.ceil((x+width)*sx)),\n            min(downloaded_size[1], math.ceil((y+height)*sy)))\n\ndef fetch_bytes(url, timeout=12):\n    request = urllib.request.Request(url, headers={\'User-Agent\': \'EcoSort-AI research dataset preparation\'})\n    with urllib.request.urlopen(request, timeout=timeout) as response:\n        return response.read()\n\nclass UnionFind:\n    def __init__(self, values):\n        self.parent = {v: v for v in values}\n    def find(self, value):\n        while self.parent[value] != value:\n            self.parent[value] = self.parent[self.parent[value]]\n            value = self.parent[value]\n        return value\n    def union(self, a, b):\n        a, b = self.find(a), self.find(b)\n        if a != b:\n            self.parent[max(a, b)] = min(a, b)\n\ndef grouped_splits(groups, seed=42):\n    """Input maps group IDs to crisp-packet presence; every group stays in one split."""\n    assignments = {}\n    for has_crisp in (False, True):\n        ids = sorted(k for k, flag in groups.items() if flag == has_crisp)\n        random.Random(seed + int(has_crisp)).shuffle(ids)\n        n = len(ids)\n        n_val = max(1, round(n*.15)) if n >= 7 else 0\n        n_test = n_val\n        for index, group_id in enumerate(ids):\n            assignments[group_id] = (\'val\' if index < n_val else\n                                      \'test\' if index < n_val+n_test else \'train\')\n    return assignments\n\ndef prepare_taco(output, annotation_bytes=None, workers=12, min_side=48):\n    """Download official preview images; keep only resolvable crops >= min_side pixels."""\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    raw = annotation_bytes if annotation_bytes is not None else fetch_bytes(ANNOTATIONS_URL)\n    annotations = json.loads(raw)\n    (output/\'annotations_source.json\').write_bytes(raw)\n    categories = {c[\'id\']: c[\'name\'] for c in annotations[\'categories\']}\n    images = {im[\'id\']: im for im in annotations[\'images\']}\n    selected = [a for a in annotations[\'annotations\'] if categories[a[\'category_id\']] in CATEGORIES]\n    by_image = defaultdict(list)\n    for ann in selected:\n        by_image[ann[\'image_id\']].append(ann)\n    cache = output/\'source_cache\'\n    cache.mkdir(exist_ok=True)\n    def download(image_id):\n        info = images[image_id]\n        path = cache/f\'image_{image_id:05d}.jpg\'\n        urls = list(dict.fromkeys(u for u in (info.get(\'flickr_640_url\'), info.get(\'flickr_url\')) if u))\n        if path.exists():\n            with Image.open(path) as im:\n                image = im.convert(\'RGB\')\n            if max(image.size) > 640:\n                image = ImageOps.contain(image, (640, 640), Image.Resampling.LANCZOS)\n                image.save(path, quality=95)\n                with Image.open(path) as im:\n                    image = im.convert(\'RGB\')\n            return image_id, image, (path.with_suffix(\'.url\').read_text() if path.with_suffix(\'.url\').exists() else \'cached URL unknown\'), None\n        failures = []\n        for url in urls:\n            for attempt in range(1):\n                try:\n                    content = fetch_bytes(url)\n                    with Image.open(io.BytesIO(content)) as im:\n                        image = ImageOps.contain(im.convert(\'RGB\'), (640, 640), Image.Resampling.LANCZOS)\n                    image.save(path, quality=95)\n                    path.with_suffix(\'.url\').write_text(url)\n                    # Return the same cached decoded pixels on initial and repeat runs.\n                    with Image.open(path) as im:\n                        return image_id, im.convert(\'RGB\'), url, None\n                except Exception as exc:\n                    failures.append(type(exc).__name__ + \': \' + str(exc))\n        return image_id, None, None, failures\n    downloaded = {}\n    errors = []\n    ids = sorted(by_image)\n    print(f\'TACO: {len(selected)} annotated candidate objects in {len(ids)} source images.\', flush=True)\n    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as pool:\n        for i, (image_id, image, url, error) in enumerate(pool.map(download, ids), 1):\n            if image is None:\n                errors.append({\'image_id\': image_id, \'errors\': error})\n            else:\n                downloaded[image_id] = (image, url)\n            if i % 25 == 0 or i == len(ids):\n                print(f\'Downloaded/checking {i}/{len(ids)} sources; failures {len(errors)}.\', flush=True)\n    # Group exact copies and near-identical whole scenes BEFORE deciding splits.\n    uf = UnionFind(sorted(downloaded))\n    fingerprints = {}\n    for image_id in sorted(downloaded):\n        image, _ = downloaded[image_id]\n        fingerprints[image_id] = (pixel_hash(image), dhash(image))\n    scene_ids = sorted(downloaded)\n    for i, a in enumerate(scene_ids):\n        for b in scene_ids[:i]:\n            if fingerprints[a][0] == fingerprints[b][0] or (fingerprints[a][1]^fingerprints[b][1]).bit_count() <= 4:\n                uf.union(a, b)\n    candidates = []\n    crop_fingerprints = {}\n    skipped = Counter()\n    for image_id in scene_ids:\n        image, url = downloaded[image_id]\n        info = images[image_id]\n        for ann in sorted(by_image[image_id], key=lambda x:x[\'id\']):\n            box = crop_box(ann[\'bbox\'], (info[\'width\'], info[\'height\']), image.size)\n            if box[2]-box[0] < min_side or box[3]-box[1] < min_side:\n                skipped[\'too_small\'] += 1\n                continue\n            crop = image.crop(box)\n            digest = pixel_hash(crop)\n            if digest in crop_fingerprints:\n                uf.union(image_id, crop_fingerprints[digest])\n                skipped[\'exact_crop_duplicate\'] += 1\n                continue\n            crop_fingerprints[digest] = image_id\n            candidates.append((image_id, ann, crop, digest, url))\n    group_flags = {}\n    for image_id, ann, crop, digest, url in candidates:\n        gid = uf.find(image_id)\n        group_flags[gid] = group_flags.get(gid, False) or categories[ann[\'category_id\']] == \'Crisp packet\'\n    split_map = grouped_splits(group_flags)\n    records = []\n    for image_id, ann, crop, digest, url in candidates:\n        split = split_map[uf.find(image_id)]\n        relative = f\'{split}/plastic/taco_{image_id:05d}_{ann["id"]:05d}.png\'\n        path = output/relative\n        path.parent.mkdir(parents=True, exist_ok=True)\n        crop.save(path)\n        records.append({\'file\':relative,\'image_id\':image_id,\'annotation_id\':ann[\'id\'],\n            \'group_id\':uf.find(image_id),\'split\':split,\'label\':\'plastic\',\n            \'source_category\':categories[ann[\'category_id\']], \'bbox\':ann[\'bbox\'],\n            \'downloaded_crop_size\':list(crop.size),\'pixel_sha256\':digest,\n            \'download_url\':url,\'original_url\':images[image_id].get(\'flickr_url\'),\n            \'license_id\':images[image_id].get(\'license\')})\n    counts = Counter(r[\'split\'] for r in records)\n    (output/\'download_errors.json\').write_text(json.dumps(errors,indent=2))\n    for split in (\'train\',\'val\',\'test\'):\n        if counts[split] < 15:\n            raise RuntimeError(f\'Too few usable TACO {split} crops ({counts[split]}). No training started. Review download_errors.json and source availability.\')\n    manifest = {\'taco_commit\':TACO_COMMIT,\'annotations_url\':ANNOTATIONS_URL,\n        \'annotations_sha256\':sha256(raw),\'candidate_objects\':len(selected),\n        \'downloaded_source_images\':len(downloaded),\'failed_source_images\':len(errors),\n        \'counts\':dict(counts),\'source_category_counts\':dict(Counter(r[\'source_category\'] for r in records)),\n        \'skip_counts\':dict(skipped),\'min_crop_side\':min_side,\'seed\':42,\n        \'split_unit\':\'source scene / exact crop duplicate / whole-image dHash distance <= 4\',\n        \'label_policy\':POLICY, \'licenses\':annotations.get(\'licenses\',[]),\'records\':records}\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    (output/\'download_errors.json\').write_text(json.dumps(errors,indent=2))\n    print(\'Usable new TACO crops:\', dict(counts), \'Total:\', len(records), flush=True)\n    # Show only training examples; leave test images and scores untouched until final evaluation.\n    sample=random.Random(42).sample([r for r in records if r[\'split\']==\'train\'],min(30,counts[\'train\']))\n    sheet=Image.new(\'RGB\',(900,6*145),\'white\')\n    draw=ImageDraw.Draw(sheet)\n    for i,r in enumerate(sample):\n        with Image.open(output/r[\'file\']) as im:\n            preview=ImageOps.contain(im,(170,115))\n        x,y=(i%5)*180,(i//5)*145\n        sheet.paste(preview,(x,y))\n        draw.text((x,y+117),f\'{r["annotation_id"]} {r["source_category"][:19]}\',fill=\'black\')\n    sheet.save(output/\'training_preview.jpg\')\n    return manifest\n\nif __name__ == \'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--output\',required=True)\n    args=parser.parse_args()\n    prepare_taco(args.output)\n', 'train_taco': '"""Train a fresh six-class classifier with TACO plastic-packaging crops."""\nimport csv\nimport hashlib\nimport json\nimport random\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom PIL import Image\nfrom taco_data import CLASSES, POLICY, pixel_hash\n\n\ndef read_trashnet(archive, destination):\n    """Extract only class images. Never overwrite or modify the uploaded archive."""\n    destination = Path(destination)\n    records = []\n    with zipfile.ZipFile(archive) as z:\n        members = []\n        for member in z.infolist():\n            parts = Path(member.filename.replace(\'\\\\\', \'/\')).parts\n            for i in range(len(parts)-2):\n                if (parts[i] in (\'train\',\'val\',\'test\') and parts[i+1] in CLASSES\n                        and len(parts[i:]) == 3 and Path(parts[-1]).suffix.lower() in (\'.jpg\',\'.jpeg\',\'.png\')):\n                    members.append((member, parts[i], parts[i+1], parts[-1]))\n                    break\n        if not members:\n            raise ValueError(\'Upload EcoSort_TrashNet_split.zip containing train/val/test class folders.\')\n        seen_paths = set()\n        for member, split, label, name in members:\n            path = destination/split/label/name\n            if str(path) in seen_paths:\n                raise ValueError(f\'Duplicate ZIP destination: {split}/{label}/{name}\')\n            seen_paths.add(str(path))\n            path.parent.mkdir(parents=True,exist_ok=True)\n            path.write_bytes(z.read(member))\n            with Image.open(path) as im:\n                digest = pixel_hash(im)\n            records.append({\'file\':str(path),\'split\':split,\'label\':label,\'source\':\'trashnet\',\'pixel_sha256\':digest})\n    for split in (\'train\',\'val\',\'test\'):\n        labels = {r[\'label\'] for r in records if r[\'split\']==split}\n        if labels != set(CLASSES):\n            raise ValueError(f\'TrashNet {split} is missing classes: {set(CLASSES)-labels}\')\n    # Remove EVERY member of groups crossing a split or label, rather than choosing a label by guesswork.\n    by_hash = defaultdict(list)\n    for r in records:\n        by_hash[r[\'pixel_sha256\']].append(r)\n    keep, excluded = [], []\n    for group in by_hash.values():\n        conflict = len({(r[\'split\'],r[\'label\']) for r in group}) > 1\n        if conflict:\n            excluded.extend(dict(r,reason=\'duplicate crossing splits or labels\') for r in group)\n        else:\n            keep.append(group[0])\n            excluded.extend(dict(r,reason=\'duplicate within class and split\') for r in group[1:])\n    return keep, excluded\n\n\ndef prepare_records(archive, taco_root, work, data_root=None):\n    work = Path(work)\n    work.mkdir(parents=True,exist_ok=True)\n    original, excluded = read_trashnet(archive,Path(data_root) if data_root else work/\'trashnet\')\n    taco_root = Path(taco_root)\n    manifest = json.loads((taco_root/\'manifest.json\').read_text())\n    original_hashes = {r[\'pixel_sha256\'] for r in original}\n    blocked = {r[\'group_id\'] for r in manifest[\'records\'] if r[\'pixel_sha256\'] in original_hashes}\n    new = []\n    groups = defaultdict(set)\n    for r in manifest[\'records\']:\n        groups[r[\'group_id\']].add(r[\'split\'])\n        row = dict(r,file=str(taco_root/r[\'file\']),source=\'taco\')\n        if r[\'group_id\'] in blocked:\n            excluded.append(dict(row,reason=\'TACO group matches TrashNet decoded pixels\'))\n        else:\n            new.append(row)\n    if any(len(splits)>1 for splits in groups.values()):\n        raise ValueError(\'TACO source group crosses splits; training stopped.\')\n    counts = Counter(r[\'split\'] for r in new)\n    if counts[\'train\']<100 or counts[\'val\']<15 or counts[\'test\']<15:\n        raise ValueError(f\'Not enough NEW wrapper images to train: {dict(counts)}\')\n    all_records = original+new\n    split_hashes = defaultdict(set)\n    for r in all_records:\n        split_hashes[r[\'pixel_sha256\']].add(r[\'split\'])\n    if any(len(s)>1 for s in split_hashes.values()):\n        raise ValueError(\'Exact image leakage detected between splits.\')\n    (work/\'excluded_images.json\').write_text(json.dumps(excluded,indent=2))\n    (work/\'dataset_records.json\').write_text(json.dumps(all_records,indent=2))\n    with open(archive, \'rb\') as handle:\n        archive_digest = hashlib.file_digest(handle, \'sha256\').hexdigest()\n    summary = {\'archive_name\':Path(archive).name,\'archive_sha256\':archive_digest,\n        \'trashnet_counts\':dict(Counter(r[\'split\'] for r in original)),\n        \'new_taco_counts\':dict(counts),\'excluded_count\':len(excluded),\'classes\':CLASSES,\n        \'class_counts_by_split\':{s:dict(Counter(r[\'label\'] for r in all_records if r[\'split\']==s)) for s in (\'train\',\'val\',\'test\')}}\n    (work/\'dataset_summary.json\').write_text(json.dumps(summary,indent=2))\n    print(\'TrashNet after duplicate exclusion:\',dict(Counter(r[\'split\'] for r in original)),flush=True)\n    print(\'NEW wrapper images:\',dict(counts),flush=True)\n    return all_records\n\n\ndef train(records, output, head_epochs=3, finetune_epochs=7, batch_size=32, device_name="cuda"):\n    import numpy as np\n    import torch\n    from torch import nn\n    from torch.utils.data import Dataset, DataLoader\n    from torchvision import models, transforms\n    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report\n    output=Path(output)\n    output.mkdir(parents=True,exist_ok=True)\n    if device_name == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU runtime in Colab before training. No training started.\')\n    random.seed(42)\n    np.random.seed(42)\n    torch.manual_seed(42)\n    torch.cuda.manual_seed_all(42)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    device=torch.device(device_name)\n    mean,std=[.485,.456,.406],[.229,.224,.225]\n    evaluation=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean,std)])\n    augmentation=transforms.Compose([transforms.RandomResizedCrop(224,scale=(.75,1.)),transforms.RandomHorizontalFlip(),\n        transforms.RandomRotation(15),transforms.ColorJitter(.15,.15,.15,.05),transforms.ToTensor(),transforms.Normalize(mean,std)])\n    class Images(Dataset):\n        def __init__(self, rows, transform): self.rows,self.transform=rows,transform\n        def __len__(self): return len(self.rows)\n        def __getitem__(self,index):\n            r=self.rows[index]\n            with Image.open(r[\'file\']) as im: image=self.transform(im.convert(\'RGB\'))\n            return image,CLASSES.index(r[\'label\'])\n    split_rows={s:[r for r in records if r[\'split\']==s] for s in (\'train\',\'val\',\'test\')}\n    generator=torch.Generator().manual_seed(42)\n    loaders={s:DataLoader(Images(rows,augmentation if s==\'train\' else evaluation),batch_size=batch_size,\n        shuffle=s==\'train\',num_workers=0,generator=generator if s==\'train\' else None) for s,rows in split_rows.items()}\n    print(\'Device:\',torch.cuda.get_device_name(0) if device_name==\'cuda\' else device_name,flush=True)\n    print(\'Starting a NEW ImageNet-initialized ResNet18. No previous EcoSort weights loaded.\',flush=True)\n    model=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)\n    model.fc=nn.Linear(model.fc.in_features,len(CLASSES))\n    model.to(device)\n    counts=Counter(r[\'label\'] for r in split_rows[\'train\'])\n    weights=torch.tensor([len(split_rows[\'train\'])/(len(CLASSES)*counts[c]) for c in CLASSES],device=device)\n    loss_fn=nn.CrossEntropyLoss(weight=weights)\n    def predict(rows):\n        model.eval()\n        truth,prediction=[],[]\n        with torch.inference_mode():\n            for x,y in DataLoader(Images(rows,evaluation),batch_size=batch_size,num_workers=0):\n                p=model(x.to(device)).argmax(1).cpu().tolist()\n                prediction.extend(p)\n                truth.extend(y.tolist())\n        return truth,prediction\n    def metrics(rows):\n        y,p=predict(rows)\n        result={\'n\':len(y),\'accuracy\':accuracy_score(y,p),\n            \'confusion_matrix\':confusion_matrix(y,p,labels=list(range(6))).tolist()}\n        if {r[\'label\'] for r in rows}=={\'plastic\'}:\n            result[\'plastic_recall\']=result[\'accuracy\']\n            result[\'note\']=\'Positive-only wrapper set: this measures recall, not six-class accuracy or plastic precision.\'\n            result[\'by_source_category\']={c:{\'n\':sum(r.get(\'source_category\')==c for r in rows),\n                \'plastic_recall\':sum(pp==4 for r,pp in zip(rows,p) if r.get(\'source_category\')==c)/sum(r.get(\'source_category\')==c for r in rows)}\n                for c in sorted({r[\'source_category\'] for r in rows})}\n        else:\n            result[\'macro_f1\']=f1_score(y,p,labels=list(range(6)),average=\'macro\',zero_division=0)\n            result[\'class_report\']=classification_report(y,p,labels=list(range(6)),target_names=CLASSES,output_dict=True,zero_division=0)\n        return result\n    history=[]\n    best=(-1.,-1.)\n    checkpoint=output/\'ecosort_resnet18_taco_plastic.pth\'\n    epoch=0\n    for stage,n_epochs in [(\'head\',head_epochs),(\'layer4\',finetune_epochs)]:\n        if stage==\'layer4\':\n            model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n        for name,param in model.named_parameters():\n            param.requires_grad=name.startswith(\'fc.\') or (stage==\'layer4\' and name.startswith(\'layer4.\'))\n        groups=[{\'params\':model.fc.parameters(),\'lr\':1e-3 if stage==\'head\' else 1e-4}]\n        if stage==\'layer4\': groups.append({\'params\':model.layer4.parameters(),\'lr\':1e-5})\n        optimizer=torch.optim.AdamW(groups,weight_decay=1e-4)\n        for _ in range(n_epochs):\n            epoch+=1\n            # Frozen layers and their batch-normalization statistics remain frozen.\n            model.eval()\n            model.fc.train()\n            if stage==\'layer4\': model.layer4.train()\n            total_loss,correct,n=0.,0,0\n            for x,y in loaders[\'train\']:\n                x,y=x.to(device),y.to(device)\n                optimizer.zero_grad(set_to_none=True)\n                logits=model(x)\n                loss=loss_fn(logits,y)\n                loss.backward()\n                optimizer.step()\n                total_loss+=loss.item()*len(y)\n                correct+=(logits.argmax(1)==y).sum().item()\n                n+=len(y)\n            val=metrics(split_rows[\'val\'])\n            wrapper_val=metrics([r for r in split_rows[\'val\'] if r[\'source\']==\'taco\'])\n            row={\'epoch\':epoch,\'stage\':stage,\'train_loss\':total_loss/n,\'train_accuracy\':correct/n,\n                \'val_accuracy\':val[\'accuracy\'],\'val_macro_f1\':val[\'macro_f1\'],\'wrapper_val_recall\':wrapper_val[\'plastic_recall\']}\n            history.append(row)\n            print(f"Epoch {epoch}/{head_epochs+finetune_epochs}: val F1 {val[\'macro_f1\']:.3f}, wrapper val recall {wrapper_val[\'plastic_recall\']:.1%}",flush=True)\n            score=(val[\'macro_f1\'],val[\'accuracy\'])\n            if score>best:\n                best=score\n                torch.save({\'model_state_dict\':model.state_dict(),\'classes\':CLASSES,\'class_names\':CLASSES,\n                    \'epoch\':epoch,\'val_macro_f1\':best[0],\'image_size\':224,\'mean\':mean,\'std\':std,\n                    \'label_policy\':POLICY,\'initialization\':\'ImageNet1K V1; no previous EcoSort checkpoint\'},checkpoint)\n    model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n    final={\'classes\':CLASSES,\'label_policy\':POLICY,\'selection\':\'highest combined validation macro F1, accuracy breaks ties\',\n        \'selected_epoch\':torch.load(checkpoint,map_location=\'cpu\',weights_only=True)[\'epoch\'],\n        \'trashnet_test\':metrics([r for r in split_rows[\'test\'] if r[\'source\']==\'trashnet\']),\n        \'new_wrapper_test\':metrics([r for r in split_rows[\'test\'] if r[\'source\']==\'taco\']),\n        \'combined_test\':metrics(split_rows[\'test\']),\n        \'limitations\':[\'TrashNet test has been inspected in earlier experiments; interpret as diagnostic.\',\n            \'TACO holdout is grouped by source and near-identical scenes, not independently collected deployment photos.\',\n            \'Packaging categories are project labels, not verified material composition or recycling advice.\',\n            \'TACO crops differ from whole-image TrashNet photos; evaluate fresh user photos before deployment.\']}\n    (output/\'metrics.json\').write_text(json.dumps(final,indent=2))\n    with (output/\'training_history.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(history[0]));writer.writeheader();writer.writerows(history)\n    print(\'Selected epoch:\',final[\'selected_epoch\'],flush=True)\n    print(\'TrashNet test accuracy:\',f"{final[\'trashnet_test\'][\'accuracy\']:.1%}",flush=True)\n    print(\'NEW wrapper test recall:\',f"{final[\'new_wrapper_test\'][\'plastic_recall\']:.1%}",flush=True)\n    return final\n', 'realwaste_data': '"""Add the six compatible RealWaste categories without changing existing splits."""\nimport hashlib\nimport io\nimport json\nimport random\nimport re\nimport urllib.request\nimport zipfile\nfrom collections import Counter,defaultdict\nfrom pathlib import Path,PurePosixPath\nfrom PIL import Image,ImageOps,ImageDraw\nfrom taco_data import CLASSES,UnionFind,dhash,pixel_hash\n\nURL=\'https://archive.ics.uci.edu/static/public/908/realwaste.zip\'\nCITATION=\'Single, S., Iranmanesh, S., & Raad, R. (2023). RealWaste. UCI Machine Learning Repository. https://doi.org/10.24432/C5SS4G\'\nMAPPING={\'Cardboard\':\'cardboard\',\'Glass\':\'glass\',\'Metal\':\'metal\',\'Miscellaneous Trash\':\'trash\',\'Paper\':\'paper\',\'Plastic\':\'plastic\'}\n\n\ndef download_realwaste(path):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    if path.exists() and zipfile.is_zipfile(path): return path\n    part=path.with_suffix(\'.zip.part\')\n    print(\'Downloading RealWaste (~657 MB).\',flush=True)\n    request=urllib.request.Request(URL,headers={\'User-Agent\':\'EcoSort-AI research\'})\n    with urllib.request.urlopen(request,timeout=60) as src,part.open(\'wb\') as dst:\n        total=0;last=0\n        while chunk:=src.read(1024*1024):\n            dst.write(chunk);total+=len(chunk)\n            if total-last>=64*1024*1024:\n                print(f\'RealWaste downloaded: {total//(1024*1024)} MiB\',flush=True);last=total\n    if not zipfile.is_zipfile(part): raise ValueError(\'RealWaste download is not a valid ZIP.\')\n    part.replace(path)\n    return path\n\n\ndef split_groups(rows,uf,seed=2026):\n    grouped=defaultdict(list)\n    for i,r in enumerate(rows): grouped[uf.find(i)].append(r)\n    strata=defaultdict(list)\n    for group,members in grouped.items():\n        # Cross-label near-duplicate groups remain intact; use a deterministic majority stratum.\n        counts=Counter(r[\'label\'] for r in members)\n        dominant=sorted(counts,key=lambda c:(-counts[c],c))[0]\n        strata[dominant].append(group)\n    result={}\n    for label in CLASSES:\n        groups=sorted(strata[label]);random.Random(seed+CLASSES.index(label)).shuffle(groups)\n        if len(groups)<7: raise ValueError(f\'Only {len(groups)} source groups for {label}; inspect grouping before training.\')\n        holdout=max(1,round(.15*len(groups)))\n        for i,group in enumerate(groups): result[group]=\'val\' if i<holdout else \'test\' if i<2*holdout else \'train\'\n    return result\n\n\ndef prepare_realwaste(archive,existing_records,output,block_size=20,near_threshold=4):\n    """Group filename-adjacent blocks and near copies before generating any split."""\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    cache=output/\'source_cache\';cache.mkdir(exist_ok=True)\n    existing_hashes={r[\'pixel_sha256\'] for r in existing_records}\n    candidates=[];skipped=[]\n    with zipfile.ZipFile(archive) as z:\n        members=[n for n in z.namelist() if PurePosixPath(n).parent.name in MAPPING and PurePosixPath(n).suffix.lower() in (\'.jpg\',\'.jpeg\',\'.png\')]\n        for i,name in enumerate(sorted(members)):\n            label=MAPPING[PurePosixPath(name).parent.name]\n            content=z.read(name)\n            with Image.open(io.BytesIO(content)) as im:\n                rgb=im.convert(\'RGB\');digest=pixel_hash(rgb);fingerprint=dhash(rgb)\n                if min(rgb.size)<96:\n                    skipped.append({\'archive_member\':name,\'reason\':\'image side below 96 pixels\'});continue\n            path=cache/f\'{label}_{PurePosixPath(name).name}\'\n            path.write_bytes(content)\n            number=re.search(r\'(\\d+)(?=\\.[^.]+$)\',name)\n            if not number: raise ValueError(f\'Cannot determine image sequence number: {name}\')\n            candidates.append({\'file\':str(path),\'label\':label,\'source\':\'realwaste\',\'archive_member\':name,\n                \'pixel_sha256\':digest,\'dhash\':fingerprint,\'sequence_block\':(int(number.group())-1)//block_size})\n            if (i+1)%500==0: print(f\'RealWaste inspected {i+1}/{len(members)} images\',flush=True)\n    uf=UnionFind(range(len(candidates)))\n    blocks={};seen={};conflicting=set();duplicates=set();collisions=set()\n    for i,r in enumerate(candidates):\n        key=(r[\'label\'],r[\'sequence_block\'])\n        if key in blocks: uf.union(i,blocks[key])\n        else: blocks[key]=i\n        digest=r[\'pixel_sha256\']\n        if digest in existing_hashes: collisions.add(i)\n        if digest in seen:\n            j=seen[digest];uf.union(i,j)\n            if r[\'label\']!=candidates[j][\'label\']: conflicting.update([i,j])\n            else: duplicates.add(i)\n        else: seen[digest]=i\n        for j,previous in enumerate(candidates[:i]):\n            if (r[\'dhash\']^previous[\'dhash\']).bit_count()<=near_threshold: uf.union(i,j)\n    # A source-group collision with an existing experiment excludes the whole new group.\n    blocked={uf.find(i) for i in collisions|conflicting}\n    assignments=split_groups(candidates,uf)\n    records=[]\n    for i,r in enumerate(candidates):\n        group=uf.find(i)\n        reason=(\'group contains existing-dataset pixels or conflicting exact labels\' if group in blocked else\n                \'exact duplicate within the same class\' if i in duplicates else None)\n        if reason: skipped.append(dict(r,reason=reason));continue\n        records.append(dict(r,group_id=f\'realwaste_{group}\',split=assignments[group]))\n    counts={s:dict(Counter(r[\'label\'] for r in records if r[\'split\']==s)) for s in (\'train\',\'val\',\'test\')}\n    for s in counts:\n        for c in CLASSES:\n            if counts[s].get(c,0)<(30 if s==\'train\' else 10):\n                raise ValueError(f\'Insufficient RealWaste {s}/{c} after grouping/exclusion: {counts[s].get(c,0)}\')\n    digest=hashlib.file_digest(open(archive,\'rb\'),\'sha256\').hexdigest()\n    manifest={\'source_url\':URL,\'source_doi\':\'https://doi.org/10.24432/C5SS4G\',\'citation\':CITATION,\n        \'license\':\'CC BY 4.0\',\'archive_sha256\':digest,\'class_mapping\':MAPPING,\n        \'unmapped_classes_excluded\':[\'Food Organics\',\'Textile Trash\',\'Vegetation\'],\n        \'candidate_images\':len(members),\'usable_images\':len(records),\'counts\':counts,\n        \'seed\':2026,\'sequence_block_size\':block_size,\'near_duplicate_dhash_threshold\':near_threshold,\n        \'split_policy\':\'Approximately 70/15/15 by grouped source blocks, stratified on majority class. Groups combine within-class filename blocks and near-identical images.\',\n        \'limitation\':\'Filename blocks are a conservative sequence heuristic, not verified capture sessions. Perceptual hashing cannot find every related scene. RealWaste comes from one landfill environment.\',\n        \'records\':records,\'excluded\':skipped}\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    validate_splits(existing_records+records)\n    preview(records,output/\'training_preview.jpg\')\n    print(\'NEW RealWaste usable images:\',len(records),flush=True)\n    print(\'NEW RealWaste split totals:\',dict(Counter(r[\'split\'] for r in records)),flush=True)\n    return manifest\n\n\ndef validate_splits(records):\n    pixels=defaultdict(set);groups=defaultdict(set)\n    for r in records:\n        pixels[r[\'pixel_sha256\']].add(r[\'split\'])\n        if \'group_id\' in r: groups[(r[\'source\'],r[\'group_id\'])].add(r[\'split\'])\n    if any(len(v)>1 for v in pixels.values()): raise ValueError(\'Exact pixels cross splits.\')\n    if any(len(v)>1 for v in groups.values()): raise ValueError(\'Source groups cross splits.\')\n\n\ndef preview(records,path):\n    sheet=Image.new(\'RGB\',(1000,840),\'white\');draw=ImageDraw.Draw(sheet)\n    for ci,label in enumerate(CLASSES):\n        pool=[r for r in records if r[\'split\']==\'train\' and r[\'label\']==label]\n        for j,r in enumerate(random.Random(42).sample(pool,min(5,len(pool)))):\n            with Image.open(r[\'file\']) as im: image=ImageOps.contain(im.convert(\'RGB\'),(190,110))\n            x,y=j*200,ci*140;sheet.paste(image,(x,y));draw.text((x,y+115),label,fill=\'black\')\n    sheet.save(path)\n', 'efficientnet_model': '"""Architecture-aware EcoSort model construction and safe checkpoint loading."""\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom torchvision import models,transforms\nfrom taco_data import CLASSES\n\n\ndef build_model(architecture,pretrained=False):\n    if architecture==\'efficientnet_b0\':\n        weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None\n        model=models.efficientnet_b0(weights=weights)\n        model.classifier[1]=nn.Linear(model.classifier[1].in_features,len(CLASSES))\n    elif architecture==\'resnet18\':\n        weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None\n        model=models.resnet18(weights=weights)\n        model.fc=nn.Linear(model.fc.in_features,len(CLASSES))\n    else: raise ValueError(f\'Unsupported architecture: {architecture}\')\n    return model\n\n\ndef load_checkpoint(path,device=\'cpu\'):\n    checkpoint=torch.load(path,map_location=device,weights_only=True)\n    if checkpoint.get(\'class_names\')!=CLASSES: raise ValueError(\'Unexpected class order.\')\n    if checkpoint.get(\'image_size\')!=224 or checkpoint.get(\'mean\')!=[.485,.456,.406] or checkpoint.get(\'std\')!=[.229,.224,.225]:\n        raise ValueError(\'Unexpected preprocessing contract.\')\n    architecture=checkpoint.get(\'architecture\')\n    if architecture is None:\n        # Earlier verified EcoSort checkpoints have no architecture metadata.\n        state=checkpoint[\'model_state_dict\']\n        if \'fc.weight\' in state and \'layer4.0.conv1.weight\' in state: architecture=\'resnet18\'\n        else: raise ValueError(\'Checkpoint has no recognizable architecture metadata.\')\n    model=build_model(architecture)\n    model.load_state_dict(checkpoint[\'model_state_dict\'],strict=True)\n    model.to(device).eval()\n    return model,checkpoint\n\n\ndef inference_transform(checkpoint):\n    return transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),\n        transforms.Normalize(checkpoint[\'mean\'],checkpoint[\'std\'])])\n', 'experiment_protocol': '"""Compare exact experiment membership independently of machine-specific paths."""\nfrom collections import Counter\nimport hashlib\nimport json\n\n\ndef membership(records):\n    return Counter((r[\'source\'],r[\'pixel_sha256\'],r[\'label\'],r[\'split\']) for r in records)\n\n\ndef require_same_split(records,reference):\n    actual,expected=membership(records),membership(reference)\n    if actual!=expected:\n        added=sum((actual-expected).values());missing=sum((expected-actual).values())\n        raise ValueError(f\'Dataset differs from the prior experiment ({added} added/changed, {missing} missing/changed). Training stopped to preserve the comparison.\')\n    canonical=sorted([list(key)+[count] for key,count in actual.items()])\n    return hashlib.sha256(json.dumps(canonical,separators=(\',\',\':\')).encode()).hexdigest()\n', 'train_efficientnet': '"""EfficientNet-B0 training with early stopping and same-set baseline comparison."""\nimport csv\nimport json\nimport random\nfrom collections import Counter\nfrom pathlib import Path\nfrom PIL import Image\nfrom taco_data import CLASSES, POLICY\nfrom efficientnet_model import build_model, load_checkpoint\n\n\ndef train(records, output, head_epochs=3, finetune_epochs=7, batch_size=32, device_name="cuda",\n          checkpoint_name="ecosort_efficientnet_b0.pth", baseline_checkpoint=None, patience=None):\n    import numpy as np\n    import torch\n    from torch import nn\n    from torch.utils.data import Dataset, DataLoader\n    from torchvision import models, transforms\n    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report\n    output=Path(output)\n    output.mkdir(parents=True,exist_ok=True)\n    if device_name == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU runtime in Colab before training. No training started.\')\n    random.seed(42)\n    np.random.seed(42)\n    torch.manual_seed(42)\n    torch.cuda.manual_seed_all(42)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    device=torch.device(device_name)\n    mean,std=[.485,.456,.406],[.229,.224,.225]\n    evaluation=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean,std)])\n    augmentation=transforms.Compose([transforms.RandomResizedCrop(224,scale=(.75,1.)),transforms.RandomHorizontalFlip(),\n        transforms.RandomRotation(15),transforms.ColorJitter(.15,.15,.15,.05),transforms.ToTensor(),transforms.Normalize(mean,std)])\n    class Images(Dataset):\n        def __init__(self, rows, transform): self.rows,self.transform=rows,transform\n        def __len__(self): return len(self.rows)\n        def __getitem__(self,index):\n            r=self.rows[index]\n            with Image.open(r[\'file\']) as im: image=self.transform(im.convert(\'RGB\'))\n            return image,CLASSES.index(r[\'label\'])\n    split_rows={s:[r for r in records if r[\'split\']==s] for s in (\'train\',\'val\',\'test\')}\n    generator=torch.Generator().manual_seed(42)\n    loaders={s:DataLoader(Images(rows,augmentation if s==\'train\' else evaluation),batch_size=batch_size,\n        shuffle=s==\'train\',num_workers=0,generator=generator if s==\'train\' else None) for s,rows in split_rows.items()}\n    print(\'Device:\',torch.cuda.get_device_name(0) if device_name==\'cuda\' else device_name,flush=True)\n    print(\'Starting a NEW ImageNet-initialized EfficientNet-B0. ResNet18 remains the comparison baseline.\',flush=True)\n    model=build_model(\'efficientnet_b0\',pretrained=True)\n    model.to(device)\n    counts=Counter(r[\'label\'] for r in split_rows[\'train\'])\n    weights=torch.tensor([len(split_rows[\'train\'])/(len(CLASSES)*counts[c]) for c in CLASSES],device=device)\n    loss_fn=nn.CrossEntropyLoss(weight=weights)\n    def predict(rows):\n        model.eval()\n        truth,prediction=[],[]\n        with torch.inference_mode():\n            for x,y in DataLoader(Images(rows,evaluation),batch_size=batch_size,num_workers=0):\n                p=model(x.to(device)).argmax(1).cpu().tolist()\n                prediction.extend(p)\n                truth.extend(y.tolist())\n        return truth,prediction\n    def metrics(rows):\n        y,p=predict(rows)\n        result={\'n\':len(y),\'accuracy\':accuracy_score(y,p),\n            \'confusion_matrix\':confusion_matrix(y,p,labels=list(range(6))).tolist()}\n        if {r[\'label\'] for r in rows}=={\'plastic\'}:\n            result[\'plastic_recall\']=result[\'accuracy\']\n            result[\'note\']=\'Positive-only wrapper set: this measures recall, not six-class accuracy or plastic precision.\'\n            result[\'by_source_category\']={c:{\'n\':sum(r.get(\'source_category\')==c for r in rows),\n                \'plastic_recall\':sum(pp==4 for r,pp in zip(rows,p) if r.get(\'source_category\')==c)/sum(r.get(\'source_category\')==c for r in rows)}\n                for c in sorted({r[\'source_category\'] for r in rows})}\n        else:\n            result[\'macro_f1\']=f1_score(y,p,labels=list(range(6)),average=\'macro\',zero_division=0)\n            result[\'class_report\']=classification_report(y,p,labels=list(range(6)),target_names=CLASSES,output_dict=True,zero_division=0)\n        return result\n    history=[]\n    best=(-1.,-1.)\n    checkpoint=output/checkpoint_name\n    best_epoch=0\n    epoch=0\n    for stage,n_epochs in [(\'head\',head_epochs),(\'features\',finetune_epochs)]:\n        if stage==\'features\':\n            model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n        for name,param in model.named_parameters():\n            param.requires_grad=name.startswith(\'classifier.\') or (stage==\'features\' and any(name.startswith(f\'features.{i}.\') for i in (6,7,8)))\n        groups=[{\'params\':model.classifier.parameters(),\'lr\':1e-3 if stage==\'head\' else 1e-4}]\n        if stage==\'features\': groups.append({\'params\':model.features[6:].parameters(),\'lr\':1e-5})\n        optimizer=torch.optim.AdamW(groups,weight_decay=1e-4)\n        stale_epochs=0\n        for _ in range(n_epochs):\n            epoch+=1\n            # Frozen layers and their batch-normalization statistics remain frozen.\n            model.eval()\n            model.classifier.train()\n            if stage==\'features\':\n                for block in model.features[6:]: block.train()\n            total_loss,correct,n=0.,0,0\n            for x,y in loaders[\'train\']:\n                x,y=x.to(device),y.to(device)\n                optimizer.zero_grad(set_to_none=True)\n                logits=model(x)\n                loss=loss_fn(logits,y)\n                loss.backward()\n                optimizer.step()\n                total_loss+=loss.item()*len(y)\n                correct+=(logits.argmax(1)==y).sum().item()\n                n+=len(y)\n            val=metrics(split_rows[\'val\'])\n            wrapper_val=metrics([r for r in split_rows[\'val\'] if r[\'source\']==\'taco\'])\n            row={\'epoch\':epoch,\'stage\':stage,\'train_loss\':total_loss/n,\'train_accuracy\':correct/n,\n                \'val_accuracy\':val[\'accuracy\'],\'val_macro_f1\':val[\'macro_f1\'],\'wrapper_val_recall\':wrapper_val[\'plastic_recall\']}\n            history.append(row)\n            print(f"Epoch {epoch}/{head_epochs+finetune_epochs}: val F1 {val[\'macro_f1\']:.3f}, wrapper val recall {wrapper_val[\'plastic_recall\']:.1%}",flush=True)\n            score=(val[\'macro_f1\'],val[\'accuracy\'])\n            if score>best:\n                best=score\n                best_epoch=epoch\n                stale_epochs=0\n                torch.save({\'model_state_dict\':model.state_dict(),\'classes\':CLASSES,\'class_names\':CLASSES,\n                    \'epoch\':epoch,\'val_macro_f1\':best[0],\'image_size\':224,\'mean\':mean,\'std\':std,\n                    \'architecture\':\'efficientnet_b0\',\'preprocessing\':\'RGB; resize 224x224; ImageNet normalization\',\n                    \'label_policy\':POLICY,\'initialization\':\'ImageNet1K V1; no previous EcoSort checkpoint\'},checkpoint)\n            else:\n                stale_epochs+=1\n            if stage==\'features\' and patience is not None and stale_epochs>=patience:\n                print(\'Early stopping: validation selection score has not improved.\',flush=True)\n                break\n    model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)[\'model_state_dict\'])\n    source_rows={s:[r for r in split_rows[\'test\'] if r[\'source\']==s] for s in sorted({r[\'source\'] for r in split_rows[\'test\']})}\n    source_test={s:metrics(rows) for s,rows in source_rows.items()}\n    baseline_test=None\n    if baseline_checkpoint is not None:\n        candidate=model\n        model,base=load_checkpoint(baseline_checkpoint,device=device)\n        baseline_test={s:metrics(rows) for s,rows in source_rows.items()}\n        del model,base\n        model=candidate\n    final={\'architecture\':\'efficientnet_b0\',\'classes\':CLASSES,\'label_policy\':POLICY,\'selection\':\'highest combined validation macro F1, accuracy breaks ties\',\n        \'selected_epoch\':torch.load(checkpoint,map_location=\'cpu\',weights_only=True)[\'epoch\'],\n        \'trashnet_test\':source_test[\'trashnet\'],\n        \'new_wrapper_test\':source_test[\'taco\'],\n        \'source_test\':source_test,\'baseline_source_test\':baseline_test,\n        \'combined_test\':metrics(split_rows[\'test\']),\n        \'limitations\':[\'All three existing test splits have now been inspected in earlier experiments; interpret as diagnostic.\',\n            \'A fresh independent evaluation set is needed before deployment claims.\',\n            \'TACO holdout is grouped by source and near-identical scenes, not independently collected deployment photos.\',\n            \'Packaging categories are project labels, not verified material composition or recycling advice.\',\n            \'TACO crops differ from whole-image TrashNet photos; evaluate fresh user photos before deployment.\']}\n    (output/\'metrics.json\').write_text(json.dumps(final,indent=2))\n    with (output/\'training_history.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(history[0]));writer.writeheader();writer.writerows(history)\n    print(\'Selected epoch:\',final[\'selected_epoch\'],flush=True)\n    print(\'TrashNet test accuracy:\',f"{final[\'trashnet_test\'][\'accuracy\']:.1%}",flush=True)\n    print(\'NEW wrapper test recall:\',f"{final[\'new_wrapper_test\'][\'plastic_recall\']:.1%}",flush=True)\n    if \'realwaste\' in source_test:\n        print(\'NEW RealWaste test accuracy:\',f"{source_test[\'realwaste\'][\'accuracy\']:.1%}",flush=True)\n        if baseline_test is not None:\n            print(\'Previous model on SAME RealWaste test:\',f"{baseline_test[\'realwaste\'][\'accuracy\']:.1%}",flush=True)\n    return final\n'}
for name,text in MODULES.items(): (source/f'{name}.py').write_text(text)
sys.path.insert(0,str(source))
for name in MODULES: sys.modules.pop(name,None)
from taco_data import prepare_taco, pixel_hash
from train_taco import prepare_records
from realwaste_data import download_realwaste, prepare_realwaste, validate_splits
from experiment_protocol import require_same_split
from efficientnet_model import load_checkpoint
from train_efficientnet import train

def locate_inputs():
    archive=None; prior=None
    for path in sorted(Path('/content').glob('*.zip')):
        try:
            with zipfile.ZipFile(path) as z:
                names=z.namelist()
                if prior is None and 'Models/ecosort_resnet18_realwaste.pth' in names and 'Results/dataset_records.json' in names:
                    prior=path
                if archive is None and any('/train/cardboard/' in '/'+n.replace('\\','/') for n in names):
                    archive=path
        except zipfile.BadZipFile: continue
    return archive,prior

archive,prior=locate_inputs()
if archive is None or prior is None:
    print('Select the missing file(s). You may select both together:',flush=True)
    if archive is None: print('  EcoSort_TrashNet_split.zip')
    if prior is None: print('  EcoSort_RealWaste_Result.zip (latest completed ResNet18 bundle)')
    uploaded=files.upload();del uploaded
    archive,prior=locate_inputs()
if archive is None or prior is None:
    raise ValueError('Both the original TrashNet split ZIP and latest RealWaste result ZIP are required. Rerun and upload whichever is missing.')
with zipfile.ZipFile(prior) as z:
    baseline=work/'verified_resnet18_baseline.pth'
    baseline.write_bytes(z.read('Models/ecosort_resnet18_realwaste.pth'))
    reference=json.loads(z.read('Results/dataset_records.json'))
    reference_rw=json.loads(z.read('Results/realwaste_manifest.json'))
    reference_trash=json.loads(z.read('Results/dataset_summary.json'))
base_model,base_info=load_checkpoint(baseline)
del base_model,base_info
print('Found original data:',archive.name,'and ResNet18 reference:',prior.name,flush=True)
result=work/f'Result_{time.time_ns()}'
(result/'Src').mkdir(parents=True)
(result/'Results').mkdir()
for name in MODULES: shutil.copy2(source/f'{name}.py',result/'Src'/f'{name}.py')
(result/"Src/predict_efficientnet.py").write_text('"""Predict a single-item photo with architecture-specific checkpoint loading."""\nimport argparse\nfrom pathlib import Path\nimport torch\nfrom PIL import Image\nfrom efficientnet_model import load_checkpoint,inference_transform\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'image\')\n    parser.add_argument(\'--model\',default=str(Path(__file__).resolve().parent.parent/\'Models/ecosort_efficientnet_b0.pth\'))\n    args=parser.parse_args()\n    model,checkpoint=load_checkpoint(args.model)\n    with Image.open(args.image) as image: x=inference_transform(checkpoint)(image.convert(\'RGB\')).unsqueeze(0)\n    with torch.inference_mode(): scores=model(x).softmax(1)[0]\n    for i in scores.argsort(descending=True).tolist(): print(f"{checkpoint[\'class_names\'][i]}: {scores[i]:.6%}")\n    print(\'Scores are model outputs, not calibrated probabilities of correctness.\')\n\n\nif __name__==\'__main__\':main()\n')
(result/"README.md").write_text('# Improve single-item classification: EfficientNet-B0 experiment\n\n## Scope\n\nEcoSort predicts one class for the whole photo: cardboard, glass, metal, paper, plastic or trash. This experiment improves that classifier; it does not add object boxes or reliably separate multiple materials in one image. An improvement is not guaranteed merely by changing the architecture or increasing its score.\n\nThe latest verified ResNet18 checkpoint selected epoch 14 and its saved results report 324/382 TrashNet, 389/522 RealWaste and 45/46 TACO packaging examples correct. On the four previously downloaded diagnostic photos it classified three as plastic and incorrectly classified the white Cheetos packet as paper with a 95.18% model score. The old TACO model classified all four as plastic. The four photos are deliberately excluded from this training experiment and cannot serve as an untouched final benchmark.\n\n## What changes\n\nThe candidate is Torchvision\'s ImageNet-initialized EfficientNet-B0. The existing three datasets and their exact class/split memberships stay fixed. Class-weighted loss, image augmentation and the project\'s RGB / 224×224 resize / ImageNet-normalization preprocessing remain explicit, so earlier and candidate predictions receive the same images. This is the project\'s preprocessing protocol, not a claim to use the pretrained weights\' default resize-and-center-crop recipe.\n\nTraining has 3 classifier-only epochs followed by up to 12 epochs training the classifier and EfficientNet feature stages 6–8. Other layers and their batch-normalization statistics remain frozen. Learning rates are 1e-3 for head warmup, then 1e-4 for the classifier and 1e-5 for the late feature stages. Early stopping allows 5 consecutive non-improving epochs in the fine-tuning stage. Selection uses combined validation macro F1, then validation accuracy for ties.\n\nAll previous ResNet checkpoints and predictors remain available. Candidate weights have architecture metadata and a distinct filename: `ecosort_efficientnet_b0.pth`. `efficientnet_model.py` builds the matching architecture; an EfficientNet checkpoint must not be loaded into a ResNet18 object.\n\n## Run in Google Colab\n\nOpen `NoteBook/EcoSort_EfficientNet_Comparison.ipynb`, select a T4 GPU, then run its one code cell. The notebook reuses available inputs where possible. If prompted, select BOTH required ZIPs together:\n\n1. `EcoSort_TrashNet_split.zip`\n2. `EcoSort_RealWaste_Result.zip` — the latest completed ResNet18 result bundle, not the earlier TACO-only result.\n\nTACO and RealWaste data caches from the earlier notebook are reused if available. Otherwise the same official data is downloaded. RealWaste is about 657 MB. The notebook reconstructs the dataset, verifies actual cached TACO crop pixels and compares source, decoded-pixel fingerprint, class and split against the previous result bundle. Any membership mismatch stops training instead of silently changing the comparison. Original input image files are never overwritten by model training. Source availability or decoding differences may cause the strict check to stop; preserve earlier source images for exact reproduction.\n\nThe expected previous split has 4,470 training, 997 validation and 950 test images. Test metrics are calculated only after validation selects a checkpoint. Both candidate and latest ResNet18 are evaluated on identical source-specific test images. These test sets have already been inspected, so results are diagnostic; collect a new independent evaluation set before deployment claims. A wrapper-only set measures recall and cannot establish plastic precision by itself.\n\n## Results and next decision\n\nDownload `EcoSort_EfficientNet_Result.zip`. It includes candidate weights, baseline comparison, selected epoch, training history, package versions, frozen membership signature, source manifests, attribution and matching inference code. The model is not automatically promoted into the app.\n\nCompare per-class errors, macro F1 and source-specific results. Test the candidate and both previous models on the same separately collected six-class photo set. A higher model score on one known example is not sufficient evidence for promotion. Keep ResNet18 if the candidate fails the intended use case.\n\nIn the inference environment after extracting the bundle into the project:\n\n```bat\npython Src\\predict_efficientnet.py "C:\\path\\to\\fresh-photo.jpg"\n```\n\n## How more data should be collected\n\nAdditional images must contain labelled garbage objects. Background-only carpet photos do not teach a six-class waste classifier which material an object belongs to. For a useful local dataset, photograph each of the six classes against several backgrounds, distances, viewing angles and lighting conditions. Include clean and crumpled objects, printed wrappers, white wrappers, transparent packaging, foil and paper as visually similar positive/negative examples. Avoid assigning every plastic example one background and every paper example another background.\n\nKeep all photographs of the same physical item or capture session together in one split. Do not treat crops, flips or augmented copies as independent new observations. Reserve a balanced evaluation set before looking at either model\'s predictions, and do not train on that set after seeing failures. Keep the current four internet photos diagnostic-only.\n\nTeam responsibilities:\n\n- Prateek: run the controlled architecture comparison, preserve results and explain selection/limits in the presentation.\n- Prince: audit training labels and collect 20–30 new original photos per class with varied objects/backgrounds. Keep source/item/session IDs and freeze an independent evaluation set. Flag ambiguous mixed-material objects rather than guessing their composition.\n- Aditya: keep the app model configurable and use an architecture-aware loader. Wait for the chosen checkpoint to be verified before switching the app.\n- Rahul: compare all candidate checkpoints on the frozen local evaluation set, including false plastic predictions, per-class F1 and app/CLI consistency.\n\n## Validation status and sources\n\nThis notebook is prepared code, not a completed GPU training run. See the PR for tests actually executed. Synthetic CPU tests do not establish trained accuracy. The previously uploaded ResNet18 checkpoint was independently loaded and its saved metrics/history checked; test scores were not independently recomputed without the original TrashNet test images.\n\n- EfficientNet-B0: https://docs.pytorch.org/vision/main/models/generated/torchvision.models.efficientnet_b0.html\n- Tan and Le (2019), EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks: https://arxiv.org/abs/1905.11946\n- TACO: Proença and Simões (2020), https://github.com/pedropro/TACO. Source categories and per-image URLs/licenses remain in the TACO manifest; toolkit licensing is not a blanket license for photographs.\n- RealWaste: Single, Iranmanesh and Raad (2023), https://doi.org/10.24432/C5SS4G, CC BY 4.0.\n- Existing label/split conventions: `Documentation/REALWASTE_EXPERIMENT.md` and `Documentation/TACO_PLASTIC_TRAINING.md`.\n')
versions={n:importlib.metadata.version(n) for n in ['torch','torchvision','Pillow','numpy','scikit-learn']}
(result/'package_versions.json').write_text(json.dumps(versions,indent=2))
with baseline.open('rb') as handle: baseline_sha=hashlib.file_digest(handle,'sha256').hexdigest()
(result/'Results/baseline_identity.json').write_text(json.dumps({'bundle':prior.name,'sha256':baseline_sha,'architecture':'resnet18'},indent=2))
try:
    with archive.open('rb') as handle: archive_sha=hashlib.file_digest(handle,'sha256').hexdigest()
    if archive_sha!=reference_trash['archive_sha256']:
        raise ValueError('The TrashNet ZIP differs from the previous experiment. Use the original split ZIP.')
    taco=Path('/content/ecosort_taco_work/taco')
    manifest_file=taco/'manifest.json'
    cached=json.loads(manifest_file.read_text()) if manifest_file.exists() else None
    if not cached or not all((taco/r['file']).exists() for r in cached['records']): prepare_taco(taco)
    records=prepare_records(archive,taco,result/'Results',data_root=work/'trashnet')
    # Cached metadata alone cannot establish that the on-disk crops are unchanged.
    for row in records:
        if row['source']=='taco':
            with Image.open(row['file']) as image: actual=pixel_hash(image)
            if actual!=row['pixel_sha256']: raise ValueError('Cached TACO crop pixels changed. Training stopped.')
    rw_archive=Path('/content/ecosort_realwaste_work/realwaste.zip')
    if not rw_archive.exists(): rw_archive=download_realwaste(work/'realwaste.zip')
    with rw_archive.open('rb') as handle: rw_sha=hashlib.file_digest(handle,'sha256').hexdigest()
    if rw_sha!=reference_rw['archive_sha256']:
        raise ValueError('RealWaste archive differs from the reference. Training stopped to preserve the experiment.')
    rw=prepare_realwaste(rw_archive,records,work/'realwaste')
    records+=rw['records']
    validate_splits(records)
    membership_sha=require_same_split(records,reference)
    from collections import Counter
    summary={'counts':dict(Counter(r['split'] for r in records)),'frozen_membership_sha256':membership_sha,
        'architecture':'efficientnet_b0','reference_bundle':prior.name,'split_membership_matches_reference':True}
    print('EXACT same dataset and splits verified:',summary['counts'],flush=True)
    (result/'Results/combined_dataset_summary.json').write_text(json.dumps(summary,indent=2))
    (result/'Results/dataset_records.json').write_text(json.dumps(records,indent=2))
    for name in ['manifest.json','download_errors.json']:
        shutil.copy2(taco/name,result/'Results'/f'taco_{name}')
    shutil.copy2(work/'realwaste/manifest.json',result/'Results/realwaste_manifest.json')
    train(records,result/'Models',head_epochs=3,finetune_epochs=12,batch_size=32,patience=5,
        baseline_checkpoint=baseline,checkpoint_name='ecosort_efficientnet_b0.pth')
except Exception:
    (result/'error.txt').write_text(traceback.format_exc())
    debug=shutil.make_archive('/content/EcoSort_EfficientNet_Debug','zip',result)
    files.download(debug)
    raise
print('Training and same-set comparison finished. Keep both old and new model bundles.',flush=True)
bundle=shutil.make_archive('/content/EcoSort_EfficientNet_Result','zip',result)
files.download(bundle)
